In [20]:
import os
import time
 
import pandas as pd
from dotenv import load_dotenv
from entsoe import EntsoePandasClient
from entsoe.mappings import Area

def fetch(query, *args, retries=3, **kwargs):
    """Call the API with retries.

    ENTSO-E returns 599 on transient server-side failures. One retry
    usually succeeds, so a pipeline must not treat it as fatal.
    """
    for attempt in range(retries):
        try:
            return query(*args, **kwargs)
        except Exception as exc:
            if attempt == retries - 1:
                raise
            wait = 2 ** attempt
            print(f"  attempt {attempt + 1} failed ({type(exc).__name__}), "
                  f"retrying in {wait}s")
            time.sleep(wait)

In [21]:
load_dotenv()
client = EntsoePandasClient(api_key=os.environ["ENTSOE_API_KEY"])

In [ ]:
SLEEP_SECONDS = 0.3

TZ = "Europe/Rome"

In [23]:
def day(date_str):
    """Return a (start, end) pair covering one calendar day.
 
    entsoe-py requires tz-aware timestamps and treats the range as
    inclusive on both ends, so the last row belongs to the next day.
    """
    start = pd.Timestamp(date_str, tz=TZ)
    return start, start + pd.Timedelta(days=1)

In [12]:
def summarise(result):
    """Describe a successful response: rows, columns, observed time step."""
    if result is None or len(result) == 0:
        return "empty"
    deltas = result.index.to_series().diff().value_counts()
    # The modal delta is the real resolution. Counting rows and dividing
    # would hide gaps, duplicates and DST days.
    step = deltas.index[0] if len(deltas) else "n/a"
    n_cols = result.shape[1] if result.ndim > 1 else 1
    return f"ok rows={len(result)} cols={n_cols} step={step}"

In [13]:
def probe(query, *args, **kwargs):
    """Run a query and return a short verdict instead of raising.
 
    A missing zone is a finding, not a reason to abort the sweep.
    """
    try:
        return summarise(query(*args, **kwargs))
    except Exception as exc:
        return type(exc).__name__
    finally:
        time.sleep(SLEEP_SECONDS)

In [ ]:
# %% Block A - availability sweep
italian_areas = [area for area in Area if area.name.startswith("IT")]
print(f"{len(italian_areas)} areas with an IT prefix\n")
 
PROBE_START, PROBE_END = day("2026-11-01")
 
rows = []
for area in italian_areas:
    record = {
        "area": area.name,
        "eic_code": getattr(area, "value", None),
        "price": probe(client.query_day_ahead_prices, area.name,
                       start=PROBE_START, end=PROBE_END),
        "load": probe(client.query_load, area.name,
                      start=PROBE_START, end=PROBE_END),
        "generation": probe(client.query_generation, area.name,
                            start=PROBE_START, end=PROBE_END),
    }
    rows.append(record)
    print(record)
 
availability = pd.DataFrame(rows).set_index("area")
print(availability.to_string())
 
# %% Block A - persist the matrix
# Keep the matrix as evidence backing docs/data_profile.md.
# Unlike console output, this is a file that can be committed.
os.makedirs("docs", exist_ok=True)
availability.to_csv("docs/area_availability.csv")
print("written to docs/area_availability.csv")

In [14]:
# ## Block B - how far back does each dataset go
assert "day" in globals(), "Setup 没在这个 kernel 跑过"

COVERAGE_ZONE = "IT_NORD"
PROBE_DATES = ["2016-06-15", "2018-06-15", "2020-06-15",
               "2021-01-15", "2022-06-15", "2024-06-15", "2026-06-15"]
 
rows = []
for date_str in PROBE_DATES:
    start, end = day(date_str)
    rows.append({
        "date": date_str,
        "price": probe(client.query_day_ahead_prices, COVERAGE_ZONE,
                       start=start, end=end),
        "load": probe(client.query_load, COVERAGE_ZONE, start=start, end=end),
        "generation": probe(client.query_generation, COVERAGE_ZONE,
                            start=start, end=end),
    })
    print(rows[-1])
 
coverage = pd.DataFrame(rows).set_index("date")
print(coverage.to_string())
 
# %% Block B - when was Calabria split out
# Calabria was separated from the southern zone in the 2021 reconfiguration,
# so queries before that date should come back empty.
for date_str in ["2020-06-15", "2021-06-15"]:
    start, end = day(date_str)
    verdict = probe(client.query_day_ahead_prices, "IT_CALA",
                    start=start, end=end)
    print(f"IT_CALA {date_str}: {verdict}")

{'date': '2016-06-15', 'price': 'ok rows=25 cols=1 step=0 days 01:00:00', 'load': 'ok rows=24 cols=1 step=0 days 01:00:00', 'generation': 'ok rows=24 cols=11 step=0 days 01:00:00'}
{'date': '2018-06-15', 'price': 'ok rows=25 cols=1 step=0 days 01:00:00', 'load': 'ok rows=24 cols=1 step=0 days 01:00:00', 'generation': 'ok rows=24 cols=11 step=0 days 01:00:00'}
{'date': '2020-06-15', 'price': 'ok rows=25 cols=1 step=0 days 01:00:00', 'load': 'ok rows=24 cols=1 step=0 days 01:00:00', 'generation': 'ok rows=24 cols=13 step=0 days 01:00:00'}
{'date': '2021-01-15', 'price': 'ok rows=25 cols=1 step=0 days 01:00:00', 'load': 'ok rows=24 cols=1 step=0 days 01:00:00', 'generation': 'ok rows=24 cols=13 step=0 days 01:00:00'}
{'date': '2022-06-15', 'price': 'ok rows=25 cols=1 step=0 days 01:00:00', 'load': 'ok rows=24 cols=1 step=0 days 01:00:00', 'generation': 'ok rows=24 cols=13 step=0 days 01:00:00'}
{'date': '2024-06-15', 'price': 'ok rows=25 cols=1 step=0 days 01:00:00', 'load': 'ok rows=24 c

In [15]:
# %% Block C - bracket the switch date
SWITCH_ZONE = "IT_NORD"
SWITCH_DATES = ["2025-06-15", "2025-09-25", "2025-10-05", "2026-11-01"]
 
rows = []
for date_str in SWITCH_DATES:
    start, end = day(date_str)
    rows.append({
        "date": date_str,
        "price": probe(client.query_day_ahead_prices, SWITCH_ZONE,
                       start=start, end=end),
        "load": probe(client.query_load, SWITCH_ZONE, start=start, end=end),
        "generation": probe(client.query_generation, SWITCH_ZONE,
                            start=start, end=end),
    })
    print(rows[-1])
 
print(pd.DataFrame(rows).set_index("date").to_string())
 
# %% Block C - the DST day
# In October the local clock repeats an hour: 25 local hours, 100 quarter-hours,
# and local timestamps that are not unique. If row count exceeds the number of
# distinct local stamps, local time cannot serve as a key.
dst_start, dst_end = day("2025-10-26")
dst_prices = client.query_day_ahead_prices("IT_NORD", start=dst_start, end=dst_end)
 
print("rows                :", len(dst_prices))
print("unique local stamps :", dst_prices.index.nunique())
print("timezone            :", dst_prices.index.tz)
print("\noffsets present:")
print(dst_prices.index.to_series().apply(lambda t: t.utcoffset()).value_counts())

{'date': '2025-06-15', 'price': 'ok rows=25 cols=1 step=0 days 01:00:00', 'load': 'ok rows=96 cols=1 step=0 days 00:15:00', 'generation': 'HTTPError'}
{'date': '2025-09-25', 'price': 'ok rows=25 cols=1 step=0 days 01:00:00', 'load': 'ok rows=96 cols=1 step=0 days 00:15:00', 'generation': 'ok rows=96 cols=15 step=0 days 00:15:00'}
{'date': '2025-10-05', 'price': 'HTTPError', 'load': 'ok rows=96 cols=1 step=0 days 00:15:00', 'generation': 'ok rows=96 cols=15 step=0 days 00:15:00'}
{'date': '2026-11-01', 'price': 'ok rows=97 cols=1 step=0 days 00:15:00', 'load': 'HTTPError', 'generation': 'ok rows=96 cols=14 step=0 days 00:15:00'}
                                             price                                    load                               generation
date                                                                                                                               
2025-06-15  ok rows=25 cols=1 step=0 days 01:00:00  ok rows=96 cols=1 step=0 days 00:15:00          

In [18]:
# %% Block D - column structure
GEN_START, GEN_END = day("2026-11-01")
 
gen_nord = fetch(client.query_generation, "IT_NORD", start=GEN_START, end=GEN_END)
time.sleep(SLEEP_SECONDS)
gen_sard = fetch(client.query_generation, "IT_SARD", start=GEN_START, end=GEN_END)
 
print("IT_NORD shape:", gen_nord.shape)
print("IT_SARD shape:", gen_sard.shape)
print("\ncolumn index nlevels:", gen_nord.columns.nlevels)
print("\nIT_NORD columns:")
for col in gen_nord.columns:
    print("  ", col)
 
# %% Block D - production types that differ between zones
nord_types = set(gen_nord.columns)
sard_types = set(gen_sard.columns)
 
print("only in NORD:")
for col in sorted(nord_types - sard_types, key=str):
    print("  ", col)
print("\nonly in SARD:")
for col in sorted(sard_types - nord_types, key=str):
    print("  ", col)
 
# %% Block D - nulls vs zeros
# A column that is entirely null is a type the TSO does not publish for this
# zone. Store it as NULL, never as 0.
print("IT_SARD null counts:")
print(gen_sard.isna().sum().to_string())
print("\nfully-null columns in IT_SARD:")
print([str(c) for c in gen_sard.columns[gen_sard.isna().all()]])

IT_NORD shape: (96, 14)
IT_SARD shape: (96, 12)

column index nlevels: 2

IT_NORD columns:
   ('Biomass', 'Actual Aggregated')
   ('Fossil Coal-derived gas', 'Actual Aggregated')
   ('Fossil Gas', 'Actual Aggregated')
   ('Fossil Hard coal', 'Actual Aggregated')
   ('Fossil Oil', 'Actual Aggregated')
   ('Hydro Pumped Storage', 'Actual Aggregated')
   ('Hydro Run-of-river and pondage', 'Actual Aggregated')
   ('Hydro Water Reservoir', 'Actual Aggregated')
   ('Solar', 'Actual Aggregated')
   ('Wind Onshore', 'Actual Aggregated')
   ('Other', 'Actual Aggregated')
   ('Energy storage', 'Actual Aggregated')
   ('Hydro Pumped Storage', 'Actual Consumption')
   ('Energy storage', 'Actual Consumption')
only in NORD:
   ('Fossil Gas', 'Actual Aggregated')
   ('Hydro Pumped Storage', 'Actual Consumption')

only in SARD:
IT_SARD null counts:
Energy storage                  Actual Consumption    38
Biomass                         Actual Aggregated      0
Fossil Coal-derived gas         Actual Ag